In [7]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
import os
import pymysql

In [10]:
#Connection to  MySQL server, parameters must be filled with correct information otherwise it won't work
connection = pymysql.connect(
    host = '127.0.0.1',
    user = 'root',
    password =  os.getenv("PASSWORD_MY_SQL"),
    database = 'data'
    )
cursor = connection.cursor()

In [11]:
sql_query = "SELECT * FROM scraped_data where location is not null and num_txs > 0;"
    

df = pd.read_sql(sql_query, connection)

/tmp/ipykernel_10887/781981850.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql_query, connection)


In [12]:
df.head()

,id,address,location,btc_format,num_txs,last_txs,entity,timezone,offsset
0,27,1FeZXDcxSzdfCN1Svz8CSX5PbJDh2neaNT,Sweden,Valid Format,11,2014-12-01,Entity 2436,Europe/Stockholm,1.0
1,30,1JXFXUBGs2ZtEDAQMdZ3tkCKo38nT2XSEp,Un-United Kingdom,Valid Format,24,2014-09-07,Entity 1827,Europe/London,0.0
2,51,1BekNv7ezkx8eAjdkrUta2BTp9bbxU9LGG,Carolina,Valid Format,3,2014-09-23,Entity 4521,None,NaN
3,203,1CHQHCENZPncS2rjiT4CEXWx7wJeq3cS2X,127.0.0.1,Valid Format,66,2025-10-12,Entity 4522,None,NaN
4,271,1E8ydXZKFLG6zHdcZoRjRT4m4AYgYYjyAi,"Amsterdam, Netherlands",Valid Format,10,2017-03-01,Entity 2497,Europe/Amsterdam,1.0


In [13]:
len(df)

11800

In [16]:
import os
import json
import time
import pandas as pd
from google import genai
from google.genai import errors # Correct import for newer SDK versions
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# 1. Extract unique locations
unique_locations = df['location'].dropna().unique().tolist()
print(f"Total unique locations to process: {len(unique_locations)}")

# 2. Initialize the client
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# 3. Process in batches to avoid 503 Payload / Token limits
batch_size = 50
location_mapping = {}
max_retries = 3
base_delay = 2

for i in range(0, len(unique_locations), batch_size):
    batch = unique_locations[i : i + batch_size]
    
    prompt = f"""
    Analyze the following geographic locations and determine their single IANA timezone.
    Strict Rules:
    1. Base your decision ONLY on the provided text.
    2. If the location is non-geographic (e.g., '127.0.0.1'), a joke (e.g., 'Un-United Kingdom'), or fictitious, output exactly: "fictitious".
    3. If the location spans multiple timezones (e.g., 'Carolina' or 'USA') or is too ambiguous, output exactly: "multiple timezones".
    4. If you genuinely cannot determine the timezone, output exactly: "not sure".
    5. Otherwise, output the exact standard IANA timezone (e.g., "Europe/Stockholm").

    Respond ONLY with a valid JSON dictionary mapping the location string to its classification.
    
    Locations to classify:
    {json.dumps(batch)}
    """
    
    for attempt in range(max_retries):
        try:
            # The correct parameter for JSON in the new Interactions API
            interaction = client.interactions.create(
                model='gemini-3.5-flash-lite',
                input=prompt,
                response_format={
                    "type": "text",
                    "mime_type": "application/json"
                }
            )
            
            # Use the interaction's convenience property
            raw_text = interaction.output_text or "{}"
            batch_mapping = json.loads(raw_text)
            
            location_mapping.update(batch_mapping)
            print(f"Processed batch {(i // batch_size) + 1} / {(len(unique_locations) // batch_size) + 1}")
            
            # Brief pause to respect normal rate limits
            time.sleep(2)
            break # Success, break out of the retry loop
            
        except errors.APIError as e: # Catch the base APIError which handles 503s safely
            if '503' in str(e) or 'high demand' in str(e).lower():
                if attempt < max_retries - 1:
                    sleep_time = base_delay * (2 ** attempt)
                    print(f"API busy (503). Retrying batch {i} in {sleep_time}s...")
                    time.sleep(sleep_time)
                else:
                    print(f"Failed batch at index {i} after {max_retries} attempts. Error: {e}")
            else:
                print(f"Unexpected API error on batch {i}: {e}")
                break
                
        except json.JSONDecodeError:
            print(f"JSON Parse Error on batch starting at index {i}. Raw text: {raw_text}")
            break # Break retry loop, move to next batch
            
        except Exception as e:
            print(f"Unexpected error on batch {i}: {e}")
            break # Break retry loop, move to next batch

# 4. Map the complete dictionary to the dataframe
df['timezone_2'] = df['location'].map(location_mapping)

# Display the comparison
print(df[['location', 'timezone', 'timezone_2']].head())

Total unique locations to process: 5111
Processed batch 1 / 103
Processed batch 2 / 103
Processed batch 3 / 103
Processed batch 4 / 103
Processed batch 5 / 103
Processed batch 6 / 103
Processed batch 7 / 103
Processed batch 8 / 103
Processed batch 9 / 103
Processed batch 10 / 103
Processed batch 11 / 103
Processed batch 12 / 103
Processed batch 13 / 103
Processed batch 14 / 103
Processed batch 15 / 103
Processed batch 16 / 103
Processed batch 17 / 103
Processed batch 18 / 103
Processed batch 19 / 103
Processed batch 20 / 103
Processed batch 21 / 103
Processed batch 22 / 103
Processed batch 23 / 103
Processed batch 24 / 103
Processed batch 25 / 103
Processed batch 26 / 103
Processed batch 27 / 103
Processed batch 28 / 103
Processed batch 29 / 103
Processed batch 30 / 103
Processed batch 31 / 103
Processed batch 32 / 103
Processed batch 33 / 103
Processed batch 34 / 103
Processed batch 35 / 103
Processed batch 36 / 103
Processed batch 37 / 103
Processed batch 38 / 103
Processed batch 39 

In [17]:
df.to_csv("users_from_bitcointalk.csv")

In [8]:
df = pd.read_csv("users_from_bitcointalk.csv")

In [10]:
len(df[df["timezone"] != df["timezone_2"]][["entity", "timezone", "timezone_2"]])

5016

In [11]:
df[df["timezone"] != df["timezone_2"]][["entity", "location", "timezone", "timezone_2"]]

,entity,location,timezone,timezone_2
1,Entity 1827,Un-United Kingdom,Europe/London,fictitious
2,Entity 4521,Carolina,NaN,multiple timezones
3,Entity 4522,127.0.0.1,NaN,fictitious
5,NaN,USA,America/New_York,multiple timezones
7,Entity 4524,USA,America/New_York,multiple timezones
9,Entity 4526,Planet Earth,NaN,fictitious
13,NaN,Canada,America/Toronto,multiple timezones
14,Entity 4537,USA,America/New_York,multiple timezones
15,Entity 4540,USA,America/New_York,multiple timezones
21,Entity 4547,"Keene, The Shire",NaN,fictitious
